# VeriGap — container-free benchmark (Kaggle)

Scores a test suite the same way the original harness does, but **without Docker**:

- **validity_rate** = (tests that pass on the reference binary) / (all tests)
- a **mutant is killed** if any reference-passing test fails on that mutant
- **mutation_score** = killed mutants / total mutants

`SUITE_SOURCE = "reference"` validates the scoring pipeline on CPU using the reference suite (no GPU needed). `SUITE_SOURCE = "gemma"` generates the suite with a model (needs GPU — added once the weekly quota resets). Data comes from the public `eth-sri/tsb-*` HF datasets.

In [ ]:
TASK = "abishekvashok__cmatrix.5c082c6"
MODEL = "google/gemma-4-e4b-it-qat-w4a16-ct"
SUITE_SOURCE = "reference"   # "reference" (CPU) or "gemma" (GPU)
import os, sys, shutil, subprocess, tempfile, gzip, hashlib, json, stat, tarfile
from pathlib import Path
import xml.etree.ElementTree as ET
WS = Path("/workspace")
print(subprocess.run(["pip","install","-q","huggingface_hub","pytest"],capture_output=True,text=True).stdout[-200:])
# make `python` resolve (some run.sh call `python`, not `python3`)
if not shutil.which("python"):
    os.symlink(shutil.which("python3"), "/usr/local/bin/python")

In [ ]:
# optional HF token (datasets are public, so this is best-effort)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret (ok, datasets are public):", type(e).__name__)

In [ ]:
from huggingface_hub import snapshot_download
impl = Path(snapshot_download("eth-sri/tsb-reference_impl", repo_type="dataset", allow_patterns=f"executable/{TASK}/**"))
REF_EXE = impl/"executable"/TASK/"executable"; REF_EXE.chmod(0o755)
mut = Path(snapshot_download("eth-sri/tsb-mutants", repo_type="dataset", allow_patterns=f"{TASK}/**"))
MUT_ROOT = mut/TASK
manifest = json.loads((MUT_ROOT/"mutants.json").read_text())
MUTANTS = manifest["mutants"] if isinstance(manifest, dict) else manifest
print("reference exe exists:", REF_EXE.exists(), "| n mutants:", len(MUTANTS))
print("sample mutant keys:", list(MUTANTS[0].keys()))

In [ ]:
# port of programbench.mutant_storage.materialize_mutant_payload (gzip + sha256, no Docker)
def materialize_mutant(m):
    payload = MUT_ROOT/m["payload"]
    if payload.is_file():
        return payload, False
    art = m.get("artifact")
    if not art:
        raise FileNotFoundError(payload)
    srcf = MUT_ROOT/art["path"]
    fd, name = tempfile.mkstemp(prefix=f"mut-{m['id']}-"); os.close(fd); tmp = Path(name)
    with gzip.open(srcf, "rb") as c, open(tmp, "wb") as o:
        shutil.copyfileobj(c, o, length=1<<20)
    h = hashlib.sha256(tmp.read_bytes()).hexdigest()
    assert h == art["sha256"], f"hash mismatch for {m['id']}"
    tmp.chmod(tmp.stat().st_mode | stat.S_IXUSR)
    return tmp, True

In [ ]:
# obtain the test suite -> SUITE_ROOT (a dir containing eval/run.sh)
if SUITE_SOURCE == "reference":
    td = Path(snapshot_download("eth-sri/tsb-reference_tests", repo_type="dataset", allow_patterns=f"tests/{TASK}.tar.gz"))
    tar = td/"tests"/f"{TASK}.tar.gz"
    ex = Path(tempfile.mkdtemp())/"suite"; ex.mkdir(parents=True)
    with tarfile.open(tar) as t: t.extractall(ex)
    run_shs = sorted(ex.rglob("run.sh"))
    print("extracted tree:", [str(p.relative_to(ex)) for p in ex.rglob('*')][:40])
    print("run.sh found:", [str(p.relative_to(ex)) for p in run_shs])
    assert run_shs, "no run.sh in reference suite"
    SUITE_ROOT = run_shs[0].parent.parent   # run.sh lives in eval/
else:
    SUITE_ROOT = Path("/kaggle/working/generated_suite")  # produced by the generation cell
print("SUITE_ROOT =", SUITE_ROOT)

In [ ]:
# container-free runner: put binary at /workspace/executable, run eval/run.sh, parse JUnit
def run_against(exe):
    if WS.exists(): shutil.rmtree(WS)
    WS.mkdir(parents=True)
    shutil.copy2(exe, WS/"executable"); (WS/"executable").chmod(0o755)
    for item in SUITE_ROOT.iterdir():
        dst = WS/item.name
        shutil.copytree(item, dst) if item.is_dir() else shutil.copy2(item, dst)
    r = subprocess.run(["bash", "eval/run.sh"], cwd=WS, capture_output=True, text=True, timeout=1800)
    res = WS/"eval"/"results.xml"
    outcomes = {}
    if res.is_file():
        for tc in ET.parse(res).getroot().iter("testcase"):
            name = (tc.get("classname","") + "::" + tc.get("name","")).strip(":")
            tags = {c.tag for c in tc}
            outcomes[name] = "skipped" if "skipped" in tags else ("failed" if ({"failure","error"} & tags) else "passed")
    return outcomes, r.returncode, (r.stdout + r.stderr)[-1200:]

In [ ]:
# validity: run the suite against the reference binary
ref_out, rc, log = run_against(REF_EXE)
print("reference run rc =", rc, "| parsed tests =", len(ref_out))
if not ref_out:
    print("NO RESULTS PARSED — run.sh tail:\n", log)
gt_pass = {n for n, s in ref_out.items() if s == "passed"}
validity = len(gt_pass)/len(ref_out) if ref_out else 0.0
print(f"validity_rate = {validity:.1%}  ({len(gt_pass)}/{len(ref_out)} passed on reference)")

In [ ]:
# mutation: a mutant is killed if any reference-passing test fails on it
killed = 0; total = 0; errors = 0
for i, m in enumerate(MUTANTS):
    total += 1
    try:
        mexe, temp = materialize_mutant(m)
    except Exception as e:
        errors += 1; print("extract-fail", m.get("id"), e); continue
    try:
        out, rc, log = run_against(mexe)
        if any(out.get(n) != "passed" for n in gt_pass):
            killed += 1
    finally:
        if temp: Path(mexe).unlink(missing_ok=True)
    if (i+1) % 10 == 0: print(f"  {i+1}/{len(MUTANTS)} mutants, {killed} killed so far")
mutation_score = killed/total if total else None
print(f"mutation_score = {mutation_score:.1%}  ({killed}/{total} killed, {errors} extract errors)")

In [ ]:
report = {"task": TASK, "suite_source": SUITE_SOURCE, "model": MODEL if SUITE_SOURCE=="gemma" else None,
          "n_tests": len(ref_out), "validity_rate": round(validity,4),
          "n_mutants": total, "n_killed": killed, "mutation_score": round(mutation_score,4) if mutation_score is not None else None}
Path("/kaggle/working/result.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## Generation (phase 2 — needs GPU)

When the weekly GPU quota resets, set `SUITE_SOURCE = "gemma"`, enable the GPU, and run the cell below before the scoring cells. It serves the model with vLLM, prompts it with the task spec using the harness's own instructions, and writes the returned files into `SUITE_ROOT`.

In [ ]:
# PHASE 2 (GPU) — placeholder; fill SPEC + wire vLLM once GPU is available.
if SUITE_SOURCE == "gemma":
    raise NotImplementedError("generation cell is added in the GPU phase")
else:
    print("SUITE_SOURCE=reference -> skipping generation")